# Distinctive terms in **Bias Definition** (contrastive TF–IDF)

Each non-empty *Bias Definition* cell is one document (focal corpus). The **background** uses the other narrative fields per row (*Topic*, *Conclusions*, *Data Details*, *Note*, *Bias Evaluation Metric*, *Debias Details*)—the shared set minus this column.

**Distinctiveness** = mean TF–IDF in focal docs minus mean TF–IDF in background docs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

_here = Path.cwd()
for _papers in (_here / "data" / "papers.csv", _here.parent / "data" / "papers.csv"):
    if _papers.is_file():
        PAPERS_CSV = _papers.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find data/papers.csv. Set the kernel cwd to the repo root (viz/) or this folder."
    )

REPO_ROOT = PAPERS_CSV.parent.parent
FOCAL_COL = "Bias Definition"
OUTPUT_DIR = REPO_ROOT / "define_bias"
CHART_PATH = OUTPUT_DIR / "distinctive_terms.png"

TEXT_COLUMNS_FOR_CONTRAST = (
    "Topic",
    "Conclusions",
    "Data Details",
    "Note",
    "Bias Evaluation Metric",
    "Bias Definition",
    "Debias Details",
)

TOP_N = 28
MIN_DF = 1
MAX_DF = 0.92
MAX_NGRAM = 1

PAPERS_CSV, FOCAL_COL, CHART_PATH

In [ ]:
def _background_text(row: pd.Series, *, exclude: set[str]) -> str:
    parts = []
    for col in TEXT_COLUMNS_FOR_CONTRAST:
        if col in exclude:
            continue
        v = row.get(col, "")
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            parts.append(s)
    return "\n".join(parts)


def distinctive_terms(
    df: pd.DataFrame,
    focal_col: str,
    *,
    top_n: int = 30,
    min_df: int = 1,
    max_df: float = 0.92,
    ngram_max: int = 1,
) -> pd.DataFrame:
    mask = df[focal_col].fillna("").astype(str).str.strip().ne("")
    focal_docs = df.loc[mask, focal_col].astype(str).str.strip().tolist()
    if not focal_docs:
        raise ValueError(f"No non-empty rows in column {focal_col!r}.")

    ex = {focal_col}
    background_docs = [_background_text(row, exclude=ex) for _, row in df.iterrows()]
    background_docs = [t if t.strip() else "(empty)" for t in background_docs]

    corpus = focal_docs + background_docs
    n_focal = len(focal_docs)

    vectorizer = TfidfVectorizer(
        stop_words="english",
        min_df=min_df,
        max_df=max_df,
        ngram_range=(1, min(ngram_max, 2)),
        sublinear_tf=True,
    )
    X = vectorizer.fit_transform(corpus)
    feature_names = np.array(vectorizer.get_feature_names_out())

    X_pos = X[:n_focal]
    X_neg = X[n_focal:]
    mean_pos = np.asarray(X_pos.mean(axis=0)).ravel()
    mean_neg = np.asarray(X_neg.mean(axis=0)).ravel()
    delta = mean_pos - mean_neg

    order = np.argsort(-delta)[:top_n]
    return pd.DataFrame(
        {
            "term": feature_names[order],
            "mean_tfidf_focal": mean_pos[order],
            "mean_tfidf_background": mean_neg[order],
            "distinctiveness": delta[order],
        }
    )

In [ ]:
df = pd.read_csv(PAPERS_CSV)
n_focal = int(df[FOCAL_COL].fillna("").astype(str).str.strip().ne("").sum())
result = distinctive_terms(
    df,
    FOCAL_COL,
    top_n=TOP_N,
    min_df=MIN_DF,
    max_df=MAX_DF,
    ngram_max=MAX_NGRAM,
)

display(
    result.style.format(
        {
            "mean_tfidf_focal": "{:.4f}",
            "mean_tfidf_background": "{:.4f}",
            "distinctiveness": "{:.4f}",
        }
    ).hide(axis="index")
)

In [ ]:
plot_df = result.iloc[::-1].reset_index(drop=True)
sns.set_theme(style="whitegrid", context="notebook")
fig_h = max(4.0, 0.38 * len(plot_df))
fig, ax = plt.subplots(figsize=(10, fig_h))
ax.barh(plot_df["term"], plot_df["distinctiveness"], color="#4c72b0", height=0.65)
ax.set_xlabel(
    f"Distinctiveness (mean TF–IDF in {FOCAL_COL} − mean in other narrative fields)",
    labelpad=8,
)
ax.set_title(f"Terms most distinctive of {FOCAL_COL} (n={n_focal} papers with text)", pad=12)
plt.tight_layout()
CHART_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(CHART_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {CHART_PATH}")